In [ ]:
# Run from the repo without installing: `pip install -e ..` makes this unnecessary.
import sys, pathlib
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import gc
import time

import numpy as np
import numpy.fft as fft
import matplotlib.pyplot as plt
from scipy.sparse.linalg import LinearOperator, lgmres
from tqdm import tqdm

import fastbox
from fastbox.box import CosmoBox, default_cosmo
from fastbox.foregrounds import ForegroundModel

from imgibbs import (
    Us, Uf, construct_A, construct_b, construct_preconditioner,
    signal_covariance_sampler as SCS,
    foreground_covariance_sampler as FCS,
    bin_it, kbins_from_crop,
    survey_grid, load, load_l2021_cube,
)


## Grid configuration

Everything downstream keys off this cell: the crop that defines the grid, the
`box_dims` that gives it a physical scale, the number of radial *k* bins, and
the signal-covariance starting point.

`box_dims` is **derived from the cropped shape**, not hardcoded, so it cannot
drift out of step with the cube. If you change `CROP`, the geometry follows.

Regenerating `S_starting_point_cropped.npy` for a different crop is
`Fastbox_Gen_Cropped.ipynb` -- it must be given the same `box_dims`, since the
voxel-to-*k*-bin mapping is only invariant under an overall rescaling for a
*cubic* box.

In [ ]:
# ---- The crop ------------------------------------------------------------
# The MeerKLASS L2021 cube is (133, 73, 500), but the drift-scan footprint is a
# diagonal band filling only 19.2% of it. CROP is the tight bounding box of that
# band. It discards ZERO valid voxels (930,938 either way) while raising the fill
# fraction to 59.1%, so the sampler inpaints flagged pixels rather than vast
# empty padding. Bounds are the min/max non-zero pixel along each spatial axis.
#
# The third slice cuts the frequency axis: 250 channels is the current working
# grid, slice(None) would use the full 500-channel band.
CROP = (slice(33, 103), slice(14, 59), slice(0, 250))     # -> (70, 45, 250)

data_cube = load_l2021_cube()[CROP]
shape     = data_cube.shape
S         = load('S_starting_point_cropped.npy')

# n_k_bins is a CEILING, not the count. kbins_from_crop lowers it until every
# bin clears min_modes, so it adapts to the crop. A hand-tuned 12 gave bin 1
# only 4 modes on the 250-channel cut, all of them kz=0 -- frequency-constant,
# and so degenerate with the foreground.
n_k_bins = 5

# ---- Geometry for THIS grid ----------------------------------------------
# Derived from CROP by imgibbs.grid, which is what 1_generate_signal_cube and
# 3_pca_transfer_function use too -- so box_dims cannot drift between the
# notebook that builds S and the notebook that consumes it. Voxels come out
# ~8.6 Mpc transverse against ~1 Mpc radial; a cubic box_dims (the old
# (232, 232, 232)) mismaps every mode to the wrong |k| bin.
grid     = survey_grid(CROP, shape)
box_dims = grid.box_dims

print(f'data_cube : {shape}   fill {(data_cube != 0).mean() * 100:.2f}%')
print(grid.summary())


In [ ]:
"""
Turn sampling for each component on/off and set the number of samples to be taken
All false and N_samples = 1 for MAP solution
"""

s_samp = True # Sampling for the 21cm signal modes
f_samp = True # Sampling for the foreground modes
d_samp = True # Sampling of the data
S_samp = True # Sampling the signal covariance
F_samp = True # Sampling the foreground covariance

flagging = True # Whether we want to emulate RFI flagging in the data

''' Stuff for testing (ignore) '''
use_true_f = False 
use_true_f_for_Fi = False
use_true_s_for_Si = False

''' Turn FG on or off '''
FG_off_on = 1 # Set to zero for no foregrounds, if you want to test the code on signal-only data

''' Where to save the data, and its naming pattern '''
out_folder = 'outputs/'
name_suffix = '_250_pk_check_'

## Loading the Data Cube and FG Model Cube

In [ ]:
# data_cube / box_dims / n_k_bins / S are set in the config cell at the top.

# LP_fg_model_cube.npy is on the retired 72^3 grid, so rebuild the least-squares
# Legendre foreground model straight from the cropped cube. Same basis the sampler
# uses (see Model Setup), but kept self-contained so the plots below can run first.
n_fg_modes = 20 #6                                  # matches n_modes in Model Setup

_nf    = data_cube.shape[2]
_vand  = np.polynomial.legendre.legvander(np.linspace(-1, 1, _nf), n_fg_modes - 1)
_basis = np.linalg.qr(_vand)[0].T               # (n_fg_modes, n_freq), orthonormal rows

fg_cube = ((data_cube.reshape(-1, _nf) @ _basis.T) @ _basis).reshape(data_cube.shape)

print(f'data_cube : {data_cube.shape}')
print(f'fg_cube   : {fg_cube.shape}  (rebuilt, {n_fg_modes} Legendre modes)')


In [ ]:

plt.matshow(data_cube[:,:,20],aspect='auto',vmin=3.,vmax=3.6,cmap='inferno')
plt.colorbar(label='T[mK]')

In [ ]:
print('data_cube min ', np.min(data_cube))
print('data_cube max ', np.max(data_cube))
print('data_cube mean', np.mean(data_cube))
print('fill fraction ', f'{(data_cube != 0).mean()*100:.2f}%')

print('fg_cube NaNs:', np.isnan(fg_cube).any())


In [ ]:
'''fits_file_path='/idia/projects/hi_im/raw_vis/MeerKLASS2021/self_cal/pix0.3_sigma4_count40/level6/re_cali1_round5/Nscan961_Tsky_cube_p0.3d_sigma4.0_iter2.fits'

with fits.open(fits_file_path) as L2021:
    header = L2021[0].header
    # data = L2021[0].data
    # L2021.info()

wcs=WCS(header)'''

In [ ]:
print('fg_cube  ', fg_cube.shape)
print('data_cube', data_cube.shape)


In [ ]:
shape = np.shape(data_cube)

std = 0.165 #0.023 #0.165 # Define the standard deviation of the noise


if flagging:
    flag=np.where(data_cube==0,0,data_cube)

    #print(np.unique(flag)) #checking that only the nans were changed to 0's

    flag=np.where(flag!=0,1, flag)

    #print(np.unique(flag))

    data_cube = data_cube * flag

    print(np.unique(data_cube))


## Plot a slice of data and of the FG model

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,6), dpi=300)

# slices
data_slice = data_cube[:, :,35]
fg_slice = fg_cube[:, :, 35]

# mask zeros for plotting
data_slice_masked = np.ma.masked_where(data_slice == 0, data_slice)
fg_slice_masked = np.ma.masked_where(fg_slice == 0, fg_slice)

# optional: make masked regions white
cmap = plt.cm.inferno.copy()
cmap.set_bad(color='white')

# plots
data_plot = ax[0].matshow(data_slice_masked.T, cmap=cmap)
fig.colorbar(data_plot, ax=ax[0])
fg_model_plot = ax[1].matshow(fg_slice_masked.T, cmap=cmap)
fig.colorbar(fg_model_plot, ax=ax[1], label='T[mK]')

# titles
ax[0].set_title('Calibrated Data Slice')
ax[1].set_title('Foreground Slice')

plt.tight_layout()
plt.show()

## Residuals between Calibrated data cube and FG cube

The FG cube was constructed by running a least squares fit on the calibrated data with legendre polynomials as a basis.

In [ ]:
residual = data_slice - fg_slice
fig = plt.figure(figsize=(10, 6), dpi=160)
ax = plt.subplot()

residual = (data_slice - fg_slice)
cmap = plt.cm.RdBu.copy()
residual_plot = ax.matshow(residual.T, cmap=cmap)
plt.colorbar(residual_plot, ax=ax, label='T [mK]')
ax.set_title('Residual Plot')

## Model Setup

In [ ]:
n_modes = 6
n_freq  = shape[2]

# PCA eigenvectors from frequency-frequency covariance (matches reference notebook)
full_valid = np.all(data_cube != 0, axis=2)  # pixels non-zero across all freq channels
d_valid    = data_cube[full_valid]            # (N_valid, n_freq)
print(f'PCA: using {d_valid.shape[0]} fully-valid pixels')

C = (d_valid.T @ d_valid) / d_valid.shape[0]          # (n_freq, n_freq)
eigenvalues_all, eigenvectors_all = np.linalg.eigh(C)
idx_sorted       = np.argsort(eigenvalues_all)[::-1]
eigenvalues_all  = eigenvalues_all[idx_sorted]
eigenvectors_all = eigenvectors_all[:, idx_sorted]

evecs = eigenvectors_all[:, :n_modes].T   # (n_modes, n_freq) — orthonormal rows

print(f'evecs shape: {evecs.shape}')
print(f'Orthonormality (max |evecs@evecs.T - I|): {np.abs(evecs @ evecs.T - np.eye(n_modes)).max():.2e}')
print(f'Top {n_modes} eigenvalues: {eigenvalues_all[:n_modes].round(6)}')

# PCA amplitudes and signal FFT — used downstream for shape metadata
d_2d   = data_cube.reshape(-1, n_freq)
f_true = (d_2d @ evecs.T).reshape(shape[0], shape[1], n_modes)
s_true = Us(data_cube, True)

print(f'f_true shape: {f_true.shape}')
print(f's_true shape: {s_true.shape}')


In [ ]:
# Polynomial (Legendre) modes as an alternative basis to the PCA eigenvectors above.
# Overwrites evecs/f_true in place so every downstream cell (A_flat, f_mean, the
# Gibbs sampling loop, Uf(evecs, ...), etc.) picks up the polynomial basis automatically.
n_freq = shape[2]
x_grid = np.linspace(-1, 1, n_freq)

# Legendre polynomials of degree 0..n_modes-1 evaluated on the frequency grid
poly_basis = np.polynomial.legendre.legvander(x_grid, n_modes - 1).T   # (n_modes, n_freq)

# Orthonormalize so the rows stay orthonormal, matching the PCA evecs convention
Q, _ = np.linalg.qr(poly_basis.T)
evecs = Q.T   # (n_modes, n_freq) — overwrites the PCA evecs

print(f'evecs (polynomial) shape: {evecs.shape}')
print(f'Orthonormality (max |evecs@evecs.T - I|): {np.abs(evecs @ evecs.T - np.eye(n_modes)).max():.2e}')

# Polynomial-mode amplitudes — overwrites f_true; s_true is unchanged since it
# doesn't depend on the basis (Us(data_cube, True) is just the data's rfft)
d_2d   = data_cube.reshape(-1, n_freq)
f_true = (d_2d @ evecs.T).reshape(shape[0], shape[1], n_modes)

print(f'f_true (polynomial) shape: {f_true.shape}')


## Noise Covariance and Weighting

In [ ]:
#N = np.ones(np.prod(shape)) * (std)**2 # std = sigma_rms

# Wang et al. (2021) Table 1 quotes ~16 K for the MeerKAT L band. Earlier runs
# used 30 K; the switch lowered the noise by (30/16)^2 ~ 3.5x, which materially
# changes the sampler's weighting between data and prior. Change it deliberately.
T_sys  = 16                                # K

# True channel width: the L band spans 856-1712 MHz over 4096 channels, i.e.
# 0.208984 MHz -- not the 0.2 MHz the paper quotes as a round number.
del_nu = (1712.0 - 856.0) / 4096 * 1e6     # Hz
del_t  = 1000                              # seconds

N = (T_sys**2) / (del_nu * del_t)

N_inv = 1/N # Invert the noise covariance matrix for the linear equation. Since N is diagonal, we can just do 1/N

# Downweight pixels corresponding to flagged regions (if frequency channel flagging is set to True)
if flagging:
    w = flag.flatten()
else:
    w = np.ones(np.prod(shape))

Nw_inv = (1/N)*w

print(f'channel width : {del_nu/1e6:.6f} MHz')
print(f'N             : {N:.4e} K^2   (N_inv = {N_inv:.4e})')


In [ ]:
print(N_inv)

In [ ]:
''' Define array shapes used when packing/unpacking the joint solution vector x.
    The solution vector x is a single 1D array that concatenates three components:
      x = [ Re(s), Im(s), f ]
    where s is the signal in Fourier space (complex, so split into real & imaginary parts)
    and f is the foreground PCA amplitudes (real-valued).
    rfft_len / rfft_shape: length and shape of the signal Fourier modes
    f_len / f_shape: length and shape of the foreground amplitude array '''

rfft_len = np.shape(s_true.flatten())[0]
rfft_shape = np.shape(s_true)
f_len = np.shape(f_true.flatten())[0]
f_shape = np.shape(f_true)

## Provide Inputs for the LHS of the linear equation $A x=b $

In [ ]:
def A_flat(x):
    ''' Compute the matrix-vector product A @ x without ever forming A explicitly.
        construct_A unpacks x into its signal (s) and foreground (f) components,
        applies the block matrix operations (see markdown cell above), and returns
        the result as a single flattened vector.
        This function is passed to LinearOperator so scipy's iterative solvers can use it. '''
    Ax = construct_A(x, S, Nw_inv, F, w, evecs, rfft_len, rfft_shape, f_len, f_shape, shape)

    return Ax.flatten()

## Sampler Starting Points

## Starting points for $s$ (HI signal in Fourier Space) and $f$ (Foreground Amplitudes in each pixel)

In [ ]:
# Signal prior mean & starting point. We assume a zero-mean prior for the signal field,
# which encodes our expectation that the 21cm brightness temperature fluctuations
# are centred on zero (no preferred direction in Fourier space).
s_mean = np.zeros(np.shape(s_true)) 
print(s_mean.shape)

f_offset = np.random.normal(loc=1.0,scale=0.0,size=f_shape)
f_true_offset = f_true*f_offset
f_mean = f_true_offset


# x0: The starting values for our sampling.
# The joint vector x concatenates [Re(s), Im(s), f] into a single 1D array.
# The signal s is complex-valued in Fourier space, so we split it into real and
# imaginary parts to keep x purely real (required by the real-valued linear solver).
x0 = np.concatenate([s_mean.real.flatten(),s_mean.imag.flatten(),
                 f_mean.flatten()])

# Set x to x0. x gets updated with each Gibbs iteration
x = x0

In [ ]:
# PCA modes are orthonormal, so d @ evecs.T is the exact LS projection
d_2d   = data_cube.reshape(-1, n_freq)
f_mean = (d_2d @ evecs.T).reshape(shape[0], shape[1], n_modes)
print(f'f_mean shape: {f_mean.shape}')
print(f'f_mean mode-0 range: {f_mean[:,:,0].min():.3f} – {f_mean[:,:,0].max():.3f} K')


In [ ]:
print(np.unique(f_offset))
print(np.shape(f_offset))
print(f_true_offset.shape)

## Starting point for the signal covariance S

### Finding S

In [ ]:
# S itself is built in 1_generate_signal_cube.ipynb and loaded in the config
# cell at the top. What is needed here is the k-binning S is indexed by, and
# it must be the SAME call on the SAME cube and box_dims, or S is attached to
# the wrong wavenumbers.

#s_offset_cube = fastbox_cube 

# Define k-bins: n_k_bins sets the number of radial bins in |k|-space.
# define_bins returns the bin centres (sig_k) and the bin index for each voxel (idxs).
#n_k_bins = 14
#box_dims = (232, 232, 232)   # Mpc — matches reference notebook
#box_dims = (1e3/0.678*(72/256), 1e3/0.678*(72/256), 1*925/0.678*(72/256))  # wrong: gives only 2 modes in bin 1 with 14 bins
# was make_kbins(shape, n_k_bins, box_dims=box_dims), which took k_min
# from the BOX. The cube is a bounding box round a diagonal band filling
# ~59% of it, so modes longer than the band are set by the zero padding,
# not by data. kbins_from_crop measures the footprint from the mask and
# takes k_min from its short axis, drops the kz=0 plane, and picks the
# bin count from occupancy. All of it follows the crop automatically.
sig_k, idxs, kbin_meta = kbins_from_crop(data_cube, box_dims, max_bins=n_k_bins)
n_k_bins = kbin_meta['n_k_bins']  # the count actually chosen


# Group the perturbed Fourier modes by their k-bin, then pass to the signal covariance sampler (SCS).
# SCS computes sum(|s_k|^2) in each bin and draws a P(k) value from the inverse-gamma posterior.
# S is the full diagonal covariance vector (one entry per voxel), PkSample is the binned P(k).
# Note: FFT s_offset_cube only once here — do NOT pre-FFT it above (point 2 fix).
# binned_s, k_bins = bin_it(np.fft.fftn(s_offset_cube-np.mean(s_offset_cube),norm='ortho'),sig_k, idxs)
# _, PkSample = SCS(np.concatenate(binned_s),np.concatenate(k_bins))

# Rebuild S from PkSample using bin indices (point 1 fix: correct voxel-to-bin mapping)
#S = np.zeros(len(idxs))
# S = np.full(data_cube.flatten().shape , 1.5)
#
# S[0]=1e30
# print(S)
# unique_bins_s = np.unique(idxs)
# unique_bins_s = unique_bins_s[unique_bins_s > 0]
# for gg, bin_idx in enumerate(unique_bins_s):
#     S[idxs == bin_idx] = PkSample[gg]
# S[idxs == 0] = 1e30  # DC mode: large variance = uninformative prior

# Pk_init = PkSample

### Loading S from fastbox_gen.ipynb

In [ ]:
# Number of Fourier modes per k-bin
unique_idxs = np.unique(idxs)
unique_idxs = unique_idxs[unique_idxs > 0]
n_modes_per_bin = np.array([np.sum(idxs == b) for b in unique_idxs])

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.bar(range(len(n_modes_per_bin)), n_modes_per_bin, color='steelblue', edgecolor='k', lw=0.5)
#for i, n in enumerate(n_modes_per_bin):
#    ax.text(i, n + max(n_modes_per_bin)*0.01, f'{n}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('k-bin index', fontsize=12)
ax.set_ylabel('Number of modes', fontsize=12)
ax.set_title('Number of Fourier modes per k-bin', fontsize=13)
ax.set_xticks(range(len(n_modes_per_bin)))
ax.set_xticks(range(len(n_modes_per_bin)))
ax.set_yscale('log')
ax.set_xticklabels([f'{i}\n({sig_k[i]:.3f})' for i in range(len(n_modes_per_bin))], fontsize=5)
ax.set_xlabel('k-bin index (k value)', fontsize=12)
plt.tight_layout()

## Starting point for F 

In [ ]:
# Foreground covariance starting point. Perturb the least-squares amplitudes
# slightly before the inverse-Wishart draw: at f == f_mean exactly the scatter
# matrix is singular, so FCS needs the spread.
f_init = f_mean * np.random.normal(1.0, 0.05, f_mean.shape)
F = np.diag(FCS(f_init.reshape(-1, n_modes)))


In [ ]:
print(F)

## Define Linear Operator

In [ ]:
L = LinearOperator(matvec=A_flat,rmatvec=A_flat,shape=(len(x0),len(x0)))

In [ ]:
construct_A(x0*0, S, Nw_inv, F, w, evecs, rfft_len, rfft_shape, f_len, f_shape, shape)

## Sampling 

In [ ]:
print( np.unique(construct_A(x0*0, S, Nw_inv, F, w, evecs, rfft_len, rfft_shape, f_len, f_shape, shape)))

In [ ]:
N_inv_scalar = 1.0 / N

In [ ]:
print(N_inv_scalar)

In [ ]:
''' The number of Gibbs samples to be drawn (usually use at least a few thousand) '''
N_samples = 100

start = time.time()

# Build the preconditioner (once per iteration, after S and F are updated)
N_inv_scalar = 1.0 / N
'''update - add shape to the end'''
precond_apply = construct_preconditioner(S, N_inv_scalar, F, evecs, rfft_len, rfft_shape, f_len, f_shape, shape)
total_len = 2 * rfft_len + f_len
M_inv = LinearOperator((total_len, total_len), matvec=precond_apply)

for rr in tqdm(range(N_samples)):


    x0 = x # Use the previous sample as the starting guess for the iterative solver (warm-starting)

    np.save(arr=x,file= out_folder + '/samples/x_sample_' + str(rr) + name_suffix + '.npy') # Save the sample for x

    ''' Generate the omega (ω) random vectors for the constrained realisation.
        These are i.i.d. standard normal draws that, when inserted into the RHS vector b,
        transform the linear solve from finding the MAP solution into drawing a sample
        from the posterior. Each omega has the same shape as its corresponding component. '''
    if s_samp:
        w_real = np.random.normal(loc=0.0, scale=1, size=shape) 
        ws = np.fft.rfftn(w_real,norm='ortho').flatten()
    else:
        ws = 0  # Setting omega to zero reduces to the MAP/Wiener filter solution

    if f_samp:
        wf = np.random.normal(loc=0.0,scale=1,size=f_shape)
    else:
        wf = 0

    if d_samp:
        wd = np.random.normal(loc=0.0,scale=1,size=shape).flatten()
    else:
        wd = 0
    ''' Construct the RHS vector b.
        b includes the data term (N^{-1} d), the prior mean terms (S^{-1} s_mean, F^{-1} f_mean),
        and the omega terms that provide the stochastic scatter for posterior sampling. '''
    ''' update add shape'''
    b = construct_b(S, N_inv, F, w, s_mean, f_mean, evecs, data_cube, ws, wf, wd, shape)

    ''' Solve Ax = b using an iterative Krylov subspace method (LGMRES).
        tol sets the convergence tolerance. The solver uses the previous sample (x0)
        as a warm start, which typically leads to faster convergence since consecutive
        samples are correlated. '''
    tol = 1e-6

    M_inv = LinearOperator((total_len, total_len), matvec=precond_apply)

    x, exit_code = lgmres(L, b.flatten(),x0=x0,rtol=tol,atol=0,M=M_inv)
    #x, exit_code = bicgstab(L, b.flatten(),x0=x0,rtol=tol,atol=0,M=M_inv)
    #x, exit_code = minres(L, b.flatten(),x0=x0,rtol=tol,M=M_inv)

    #print(rr)

    ''' Sample for S using the signal covariance sampler.
        This is the Gibbs step for the power spectrum: given the current signal sample s,
        draw a new P(k) from the inverse-gamma conditional posterior in each k-bin. '''
    if S_samp:

        # Unpack the signal from x: first rfft_len entries are Re(s), next are Im(s)
        s = x[0:rfft_len].reshape(rfft_shape) + x[rfft_len:2*rfft_len].reshape(rfft_shape)*1j

        # Transform back to real space (inverse FFT), subtract the mean,
        # then FFT again to get mean-subtracted Fourier modes for power spectrum estimation
        s = Us(s,False) # Inverse real-valued Fourier transform
        s = np.fft.fftn(s - np.mean(s),norm='ortho')

        # Bin the Fourier modes by |k| and draw a new power spectrum sample
        binned_s, k_bins = bin_it(s, sig_k, idxs) 
        _, PkSample = SCS(np.concatenate(binned_s),np.concatenate(k_bins))

        # Rebuild S from PkSample using bin indices (point 1 fix: correct voxel-to-bin mapping)
        S = np.zeros(len(idxs))
        unique_bins_s = np.unique(idxs)
        unique_bins_s = unique_bins_s[unique_bins_s > 0]
        for gg, bin_idx in enumerate(unique_bins_s):
            S[idxs == bin_idx] = PkSample[gg]
        # idxs==0 used to be exactly ONE voxel (DC). It now also covers
        # the kz=0 plane and sub-footprint modes (~3200 voxels on the
        # 250-channel crop). A flat 1e30 prior across that plane invites
        # the signal to absorb foreground power in precisely the modes
        # where the two are degenerate, so suppress those and leave only
        # the true DC mode free to carry the overall mean.
        S[idxs == 0] = 1e-12 * np.median(PkSample)
        S[kbin_meta['dc_index']] = 1e30

        np.save(arr=PkSample,file= out_folder + '/samples/Pk_trace' + str(rr) + name_suffix + '.npy')
        np.save(arr=S,file= out_folder + '/samples/S_trace' + str(rr) + name_suffix + '.npy')

    else:
        pass

    ''' Sample for F using the foreground covariance sampler.
        This is the Gibbs step for the foreground covariance: given the current foreground
        amplitudes f, draw a new F from the inverse-Wishart conditional posterior. '''
    if F_samp:

        # Extract the foreground amplitudes from the end of the solution vector x
        x_1_recon_soln = x[2*rfft_len:2*rfft_len + f_len].reshape(f_shape)

        # Remove the imaginary part, which should be small, and should only exist because of numerical errors
        f_ms = x_1_recon_soln.real


        # Mean-subtract before computing the scatter matrix (point 6 fix).
        # At initialisation f = f_mean so this would be singular — but here in
        # the sampling loop the sampled f has diverged from f_mean.
        """ update """
        f_centered = (f_ms - f_mean).reshape((shape[0]*shape[1], n_modes))
        F = FCS(f_centered)
        F = np.diag(F)

        np.save(arr=F,file= out_folder + '/samples/F_trace' + str(rr) + name_suffix + '.npy')

    else:
        pass

    #Reconstruct the preconditioner
    """ update """
    precond_apply = construct_preconditioner(S, N_inv_scalar, F, evecs,
                                         rfft_len, rfft_shape, f_len, f_shape, shape)
    M_inv = LinearOperator((total_len, total_len), matvec=precond_apply)


    gc.collect() # Free unused memory after each iteration

end = time.time()
print((end - start))


In [ ]:
## Evaluate Results

In [ ]:
freq_idx = 55  # frequency channel to slice

fig, ax = plt.subplots(1, 2, figsize=(15,5), dpi=300)
#ax = plt.subplot(projection=wcs, slices=('x', 'y', freq_idx))

# Reconstruct model foreground cube
f_soln = x[2*rfft_len : 2*rfft_len + f_len].reshape(f_shape)
#print(np.max(f_soln))
fg_model = Uf(evecs, f_soln, False).reshape(shape)

# slices
fg_samp_slice = fg_model[:, :, freq_idx] # from the sampling process
data_slice = data_cube[:, :, freq_idx] #the original calibrated data cube

# optional masking
fg_samp_slice = np.ma.masked_where(fg_samp_slice <=0.5 , fg_samp_slice)
data_slice = np.ma.masked_where(data_slice == 0, data_slice)


# optional: make masked regions white
cmap = plt.cm.inferno.copy()
cmap.set_bad(color='white')

# overlay = ax.get_coords_overlay("icrs")
# overlay.grid(color="blue", ls="dotted")

# plots
data_plot = ax[0].matshow(data_slice.T, cmap=cmap)

fg_plot = ax[1].matshow(fg_samp_slice.T, cmap=cmap)

plt.colorbar(data_plot, ax=ax[0], label='T [K]')
plt.colorbar(fg_plot, ax=ax[1], label='T [K]')

# titles
ax[0].set_title('Calibrated Data Slice')
ax[1].set_title(f'Foreground Slice After {N_samples} Iterations')

plt.tight_layout()
#plt.savefig('MAP_FG_vs_Data_Slice_6_LPs.png')
plt.show()


In [ ]:
#Residuals 
fig = plt.figure(figsize=(10, 6), dpi=160)
ax = plt.subplot()

residual = data_slice - fg_samp_slice 
cmap = plt.cm.RdBu.copy()
residual_plot = ax.matshow(residual.T, cmap=cmap)
plt.colorbar(residual_plot, ax=ax, label='T [K]')
ax.set_title('Residual Plot')

In [ ]:
# Pixel spectrum comparison — matches reference notebook 'Evaluate Results' figure
# Frequency axis: the cube is channels CH0..CH0+n_freq of the L band
# (856-1712 MHz over 4096 channels), so the channel width is 0.208984 MHz — not 0.2.
CH0 = 550                                    # first channel of the cube in the full band
dnu = (1712.0 - 856.0) / 4096                # MHz per channel
freqs_plot = 856.0 + (CH0 + np.arange(n_freq)) * dnu   # MHz

# Find a representative pixel that is non-zero across all frequency channels
full_valid = np.all(data_cube != 0, axis=2)  # (Nx, Ny)
good = np.where(full_valid)
if len(good[0]) == 0:
    n_valid = (data_cube != 0).sum(axis=2)
    r_pix, c_pix = np.unravel_index(np.argmax(n_valid), n_valid.shape)
else:
    mid = len(good[0]) // 2
    r_pix, c_pix = int(good[0][mid]), int(good[1][mid])
print(f'Representative pixel: RA={r_pix}, Dec={c_pix}')

# Least-squares foreground fit using the same LP basis as the sampler
d_2d = data_cube.reshape(-1, n_freq)
a_ls = d_2d @ evecs.T                      # (Nx*Ny, n_modes)
fg_ls_cube = (a_ls @ evecs).reshape(shape) # (Nx, Ny, n_freq)

d_pix       = data_cube[r_pix, c_pix, :]
fg_samp_pix = fg_model[r_pix, c_pix, :].real
fg_ls_pix   = fg_ls_cube[r_pix, c_pix, :]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=130)

ax = axes[0]
ax.plot(freqs_plot, d_pix,       color='steelblue', lw=1.2, label='L-Band Data Amplitude')
ax.plot(freqs_plot, fg_samp_pix, color='red',       lw=1.5, label='Sampled FG Model Amplitude')
ax.plot(freqs_plot, fg_ls_pix,   color='green',     lw=1.5, ls='--', label='LeastSq FG Model Amplitude')
ax.set_xlabel('Frequency (MHz)', fontsize=12)
ax.set_ylabel('Amplitude in Pixel (K)', fontsize=12)
ax.set_title(f'FG model at pixel (RA={r_pix}, Dec={c_pix})', fontsize=12)
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(freqs_plot, d_pix - fg_samp_pix, color='red',   lw=1.2, label='Residual (Sampled)')
ax.plot(freqs_plot, d_pix - fg_ls_pix,   color='green', lw=1.2, label='Residual (LeastSq)')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_xlabel('Frequency (MHz)', fontsize=12)
ax.set_ylabel('Residual (K)', fontsize=12)
ax.set_title('Residuals: data − FG model', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Reconstruct model signal from the last x sample
s_soln = (x[0:rfft_len].reshape(rfft_shape)
          + x[rfft_len:2*rfft_len].reshape(rfft_shape) * 1j)

s_model = Us(s_soln, False)

fig = plt.figure(figsize=(10, 6), dpi=160)
ax = plt.subplot()  # <-- FIX: define axis

# Extract frequency slice
data_slice = s_model[:, :, freq_idx]

# optional masking
#data_slice = np.ma.masked_where(data_slice == 0, data_slice)

# plot
im = ax.imshow(data_slice.T, cmap='inferno', origin='lower')

plt.colorbar(im, ax=ax, label='T [K]')

#fig.suptitle('MAP Model Signal using LGMRES Solver')

# optional labels
ax.set_title(f'HI signal After {N_samples} Iterations', fontsize=12)
ax.set_xlabel('y')
ax.set_ylabel('x')

plt.tight_layout()
plt.show()

In [ ]:
import glob, os

# Load ONLY the traces written by this run.
#
# outputs/samples/ is never cleared between runs, so it still holds Pk traces
# from an earlier 14-k-bin run at indices 100-999. This run wrote 0-99 with 12
# bins. Globbing everything mixed 12- and 14-element arrays, which is what made
# np.array() fail with the inhomogeneous-shape error -- and left a stale 14-bin
# all_Pk in memory, so the P(k) plot below then died on "x and y must be the
# same size" against the 12-element sig_k.
_pat = out_folder + '/samples/Pk_trace*' + name_suffix + '.npy'


def _trace_idx(path):
    return int(os.path.basename(path).split('Pk_trace')[1].split(name_suffix)[0])


_found = glob.glob(_pat)
_files = sorted([p for p in _found if _trace_idx(p) < N_samples], key=_trace_idx)
_stale = len(_found) - len(_files)

all_Pk  = np.array([np.load(f) for f in _files])
N_total = len(all_Pk)

if N_total == 0:
    raise FileNotFoundError(f'no Pk traces matching {_pat}')
if N_total != N_samples:
    print(f'WARNING: expected {N_samples} traces, found {N_total}')
if all_Pk.shape[1] != len(sig_k):
    raise ValueError(
        f'traces have {all_Pk.shape[1]} k-bins but sig_k has {len(sig_k)}. '
        'outputs/samples/ was written with a different n_k_bins -- clear it and re-run.')

# Burn-in. S starts flat so iteration 0 is degenerate; 25 is where the trace has
# plateaued. Same value is used for the posterior-mean map two cells down.
cut = 10

print(f'Loaded {N_total} Pk samples, shape: {all_Pk.shape}, burn-in cut = {cut}')
if _stale:
    print(f'Ignored {_stale} stale traces (index >= {N_samples}) left over from an earlier run')


In [ ]:
# Posterior-mean signal map, averaging the post-burn-in samples.
# Iteration 0 is badly foreground-contaminated when S starts flat, so discard a
# burn-in -- check the trace has actually plateaued rather than trusting a fixed cut.
# cut comes from the loader cell above; m tracks N_samples so this stays correct
# if the sampling loop is re-run with a different length.
m = N_total

s_avg = np.zeros(shape)
for jj in range(cut, m):
    samp = np.load(out_folder + f'/samples/x_sample_{jj}' + name_suffix + '.npy')
    s_j = samp[:rfft_len].reshape(rfft_shape) + 1j*samp[rfft_len:2*rfft_len].reshape(rfft_shape)
    s_avg += Us(s_j, False).real
s_avg /= (m - cut)

print(f'averaged samples {cut}-{m-1} ({m-cut} samples)')
plt.matshow(s_avg[:, :, freq_idx].T, cmap='inferno')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)

# Simulated HI P(k) from Fastbox_Gen_Cropped.ipynb -- the truth curve.
# NOTE: the simulation has no beam applied, while the data is smoothed by the
# ~1 deg primary beam (~26.6 Mpc, about 3.3 voxels), so expect the curves to
# diverge at high k_perp for reasons unrelated to the sampler.
try:
    fastbox_Pk = np.load('Fastbox_Pk_cropped.npy')
    fastbox_k  = np.load('Fastbox_kbins_cropped.npy')
except FileNotFoundError:
    fastbox_Pk = fastbox_k = None
    print('Fastbox_Pk_cropped.npy not found -- run Fastbox_Gen_Cropped.ipynb')

# Plot each post-burn-in sample as a faint scatter point
for i in range(cut, N_total):
    ax.scatter(sig_k, all_Pk[i], color='0.75', alpha=0.5, s=4, zorder=1)
# dummy for legend
ax.scatter([], [], color='0.75', s=20, label='P(k) samples')

# Plot the posterior mean
pk_mean = np.mean(all_Pk[cut:], axis=0)
ax.plot(sig_k, pk_mean, color='#2a78d6', ls='--', marker='o', lw=2, ms=5,
        label=f'Gibbs posterior mean (samples {cut}-{N_total-1})', zorder=3)

# Plot Fastbox HI Pk
if fastbox_Pk is not None:
    ax.plot(fastbox_k, fastbox_Pk, '*-', color='#1baf7a', lw=2, ms=8,
            label='Fastbox simulated HI (no beam)', zorder=4)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'k [Mpc$^{-1}$]', fontsize=13)
ax.set_ylabel(r'P(k) [K$^2$]', fontsize=13)
ax.set_title('Model signal Pk compared to simulated HI Pk', fontsize=13)
ax.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
#plt.savefig('MAP_Pk_vs_Sim_Pk.png')
plt.show()


## Transfer function — PCA benchmark

Same method as `~/Gibbs_MWE/TransferFunction.ipynb` (Cunnington et al. 2023), ported
to the cropped (70, 45, 500) grid: PCA-clean the **data cube**, and calibrate the
signal loss by injecting mock HI into the data and re-cleaning.

$$T(k) = \frac{P\!\left(X_m^\mathrm{clean} - X^\mathrm{clean},\; X_m\right)}{P(X_m,\, X_m)}$$

The cross-power in the numerator is what keeps it unbiased — the mock is the only
field common to both, so foreground residuals and thermal noise average out instead
of contributing a positive bias.

This is a self-contained pipeline, independent of the Gibbs sampler: PCA cleaning in
place of the joint solve, $T(k)$ in place of marginalising over the foreground model.
It gives the standard-pipeline number to compare the Gibbs $P(k)$ against.

Three things are matched to the sampler so the comparison is like-for-like: the same
`n_modes = 6` foreground modes, the same `sig_k` / `idxs` radial binning, and the same
ortho-normalised FFT convention. Mocks use the same recipe as
`Fastbox_Gen_Cropped.ipynb` (lognormal + RSD, mK → K).

**Note the estimators differ.** PCA cleaning subtracts a foreground model and leaves
everything else — including the full thermal noise — in the residual. The Gibbs
$P(k)$ comes from the sampled signal field $s$, which is noise-suppressed by the
Wiener filter. So the PCA curve sitting above the Gibbs curve at high $k$ is the two
estimators behaving differently, not a disagreement about the HI.

**Note on the MWE.** There the cube is loaded as `(freq, dec, ra)`, but `pca_filter`
documents that *"the 3rd axis of the array is frequency"* — so on that 72³ cube it
cleans along RA, not frequency. The grid is cubic, so nothing errors and the output
looks plausible. Here `data_cube` is `(ra, dec, freq)`, so axis 2 really is frequency.

In [ ]:
from fastbox.filters import pca_filter
from fastbox.tracers import HITracer

# ---- Power spectrum estimators -------------------------------------------
# Same |k| binning (sig_k / idxs) and the same ortho-normalised FFT convention
# the sampler's SCS step uses, so everything here is directly comparable to the
# Gibbs PkSample traces.
_unique_bins = np.unique(idxs)
_unique_bins = _unique_bins[_unique_bins > 0]


def pk_cross(a, b):
    """Binned cross power of two real cubes."""
    A = np.fft.fftn(a - np.mean(a), norm='ortho').flatten()
    B = np.fft.fftn(b - np.mean(b), norm='ortho').flatten()
    prod = (A * np.conj(B)).real
    return np.array([prod[idxs == bb].mean() for bb in _unique_bins])


def pk_auto(a):
    return pk_cross(a, a)


# ---- Mock HI generator ----------------------------------------------------
# Identical recipe to Fastbox_Gen_Cropped.ipynb: lognormal density, RSD, and the
# mK -> K conversion. Takes ~1.3 s per realisation on this grid.
LINE_FREQ = 1420.405752                      # MHz
_dnu   = (1712.0 - 856.0) / 4096
_freqs = 856.0 + np.arange(550, 550 + n_freq) * _dnu
_z     = LINE_FREQ / _freqs - 1.0
z_mid  = 0.5 * (_z.min() + _z.max())


def mock():
    """One mock HI realisation on the cropped grid, in K."""
    box = CosmoBox(cosmo=default_cosmo, box_scale=box_dims, nsamp=shape,
                   redshift=z_mid, realise_now=False)
    box.realise_density()
    tracer   = HITracer(box)
    delta_ln = box.lognormal(box.delta_x * tracer.bias_HI())
    vel_k    = box.realise_velocity(delta_x=box.delta_x, inplace=True)
    vel_z    = np.fft.ifftn(vel_k[2]).real
    delta_s  = box.redshift_space_density(delta_x=delta_ln.real, velocity_z=vel_z,
                                          sigma_nl=120., method='linear')
    # signal_amplitude() is in mK; the data cube is in K
    return (tracer.signal_amplitude() * (1. + delta_s)) / 1000.0


# ---- PCA clean of the data cube ------------------------------------------
nmodes_pca = n_modes    # 6, the same number of foreground modes the sampler uses

# data_cube is (ra, dec, freq), and pca_filter cleans along the LAST axis, so this
# genuinely cleans in frequency. (Worth checking in the MWE, where the cube is
# loaded as (freq, dec, ra) on a cubic grid -- there axis 2 is RA.)
#
# mask_flagged: 41% of voxels are flagged to exactly zero. PCA still fits a
# foreground spectrum through partially-flagged pixels, so the cleaned cube picks
# up spurious power in the gaps -- its rms is actually higher on flagged voxels
# (0.0197 K) than on valid ones (0.0154 K). Re-masking removes that, at the cost
# of folding the mask loss into T(k) (it drops the high-k plateau from ~0.99 to
# ~0.66). False reproduces the MWE convention.
mask_flagged = False


def clean(cube):
    out = pca_filter(cube, nmodes=nmodes_pca)
    return out * flag if mask_flagged else out


cleaned_cube = clean(data_cube)
pca_pk = pk_auto(cleaned_cube)

# ---- Transfer function ----------------------------------------------------
# T(k) = P(X_m_clean - X_clean, X_m) / P(X_m, X_m), following Cunnington et al.
# (2023). The cross-power in the numerator is the point: the mock is the only
# field common to both, so foreground residuals and noise average out rather
# than adding a positive bias.
N_mocks = 100

T_s     = np.zeros((N_mocks, len(sig_k)))
mock_pk = np.zeros((N_mocks, len(sig_k)))

t0 = time.time()
for uu in tqdm(range(N_mocks), desc='mocks'):
    mock_s = mock()
    inj    = data_cube + (mock_s * flag if mask_flagged else mock_s)

    X_m_clean = clean(inj) - cleaned_cube

    mock_pk[uu] = pk_auto(mock_s)
    T_s[uu]     = pk_cross(X_m_clean, mock_s) / mock_pk[uu]

print(f'{N_mocks} mocks in {time.time()-t0:.0f} s')

T_m = np.mean(T_s, axis=0)                       # mean transfer function
corrected_PS  = pca_pk / T_m                     # TF-corrected P(k)
corrected_err = np.std(pca_pk / T_s, axis=0)     # scatter from finite N_mocks
true_pk       = np.mean(mock_pk, axis=0)         # input HI power, same estimator

# corrected_PS above divides an AUTO power by T. pca_pk carries residual
# foreground (low k) and noise (high k) as well as any surviving HI, so
# 1/T amplifies those too -- measured, corrected_PS/true sits at 4-6 in
# the lowest bins and 9-18 in the highest. And on real data, where the HI
# is not detectable, pca_pk is essentially ALL bias. So corrected_PS is
# kept for reference but must NOT be read as an HI measurement.
#
# The noise-free comparator is an INJECTION cross-power: put a known mock
# into the data, clean with and without it, and cross-correlate the
# difference against the mock. Residual foreground and noise are common to
# both cleans and cancel in the difference; what survives is uncorrelated
# with the mock and averages to zero in the cross-power.
#
# NOTE this is circular for the PCA arm -- pca_rec / mock_ref_pk is the
# definition of T_pca. Its value is that the SAME injection can be pushed
# through the Gibbs sampler, and T_gibbs vs T_pca is the non-circular
# comparison: does marginalising over the foreground lose less signal than
# projecting it out? Save mock_ref so both arms use an identical signal.
np.random.seed(1234)
mock_ref    = mock()
inj_ref     = data_cube + (mock_ref * flag if mask_flagged else mock_ref)
pca_rec     = pk_cross(clean(inj_ref) - cleaned_cube, mock_ref)
mock_ref_pk = pk_auto(mock_ref)
T_pca_ref   = pca_rec / mock_ref_pk
np.save('mock_ref_injection.npy', mock_ref)
print('\ninjection recovery (PCA arm)')
print(f'{"k":>10} {"T_pca_ref":>10} {"T_m (100 mocks)":>16}')
for i in range(len(sig_k)):
    print(f'{sig_k[i]:10.4f} {T_pca_ref[i]:10.4f} {T_m[i]:16.4f}')

print(f'\n{"k [1/Mpc]":>10} {"T(k)":>8} {"P_pca":>11} {"corrected":>11} '
      f'{"true HI":>11} {"Gibbs mean":>11}')
for i in range(len(sig_k)):
    print(f'{sig_k[i]:10.4f} {T_m[i]:8.3f} {pca_pk[i]:11.3e} {corrected_PS[i]:11.3e} '
          f'{true_pk[i]:11.3e} {pk_mean[i]:11.3e}')

In [ ]:
fig, (ax, axT) = plt.subplots(1, 2, figsize=(13, 5), dpi=200,
                              gridspec_kw={'width_ratios': [1.55, 1]})

# ---- left: power spectra ----
ax.plot(sig_k, pk_mean, color='#2a78d6', ls='--', marker='o', lw=2, ms=5, zorder=5,
        label=f'Gibbs posterior mean (samples {cut}-{N_total-1})')

# Same quantity twice -- linestyle carries the correction, not a new hue.
ax.plot(sig_k, pca_pk, color='#eb6834', ls=':', marker='s', lw=2, ms=5, mfc='none',
        zorder=3, label=f'PCA-cleaned P(k), {nmodes_pca} modes')
ax.fill_between(sig_k, corrected_PS - 2*corrected_err, corrected_PS + 2*corrected_err,
                color='#eb6834', alpha=0.2, lw=0, zorder=2)
ax.plot(sig_k, corrected_PS, color='#eb6834', ls='-', marker='s', lw=2, ms=5,
        zorder=4, label=r'PCA-cleaned $\div\,T(k)$  ($\pm2\sigma$)')

ax.plot(sig_k, true_pk, '*-', color='#1baf7a', lw=2, ms=8, zorder=6,
        label=f'True HI (mean of {N_mocks} mocks)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'k [Mpc$^{-1}$]', fontsize=12)
ax.set_ylabel(r'P(k) [K$^2$]', fontsize=12)
ax.set_title('Gibbs P(k) vs PCA + transfer function', fontsize=12)
ax.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=8.5, loc='lower left')

# ---- right: the transfer function itself ----
axT.fill_between(sig_k, T_m - 2*np.std(T_s, axis=0), T_m + 2*np.std(T_s, axis=0),
                 color='#2a78d6', alpha=0.2, lw=0)
axT.plot(sig_k, T_m, color='#2a78d6', marker='o', lw=2, ms=5, zorder=3)
axT.axhline(1.0, color='0.4', lw=0.9, ls='--', zorder=1)
axT.annotate('no signal loss', xy=(1.0, 1.0), xycoords=('axes fraction', 'data'),
             xytext=(-2, 4), textcoords='offset points',
             ha='right', va='bottom', fontsize=8, color='0.4')

axT.set_xscale('log')
axT.set_ylim(0, 1.15)
axT.set_xlabel(r'k [Mpc$^{-1}$]', fontsize=12)
axT.set_ylabel(r'$T(k)$', fontsize=12)
axT.set_title(f'Transfer function ({N_mocks} mocks, $\\pm2\\sigma$)', fontsize=12)
axT.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    axT.spines[side].set_visible(False)

plt.tight_layout()
#plt.savefig('Pk_gibbs_vs_pca_tf.png', bbox_inches='tight')
plt.show()

# ---- ratio to truth, as in the MWE's final figure ----
fig, ax = plt.subplots(figsize=(8, 4), dpi=200)
ax.plot(sig_k, pca_pk / true_pk, color='#eb6834', ls=':', marker='s', ms=4,
        mfc='none', lw=1.8, label='PCA-cleaned / True')
ax.plot(sig_k, corrected_PS / true_pk, color='#eb6834', ls='-', marker='s', ms=4,
        lw=1.8, label='Corrected / True')
ax.plot(sig_k, pk_mean / true_pk, color='#2a78d6', ls='--', marker='o', ms=4,
        lw=1.8, label='Gibbs posterior mean / True')
ax.axhline(1, color='0.4', ls='--', lw=1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=12)
ax.set_ylabel(r'$P(k)\,/\,P_\mathrm{true}(k)$', fontsize=12)
ax.set_title('Residual bias against the input HI power', fontsize=12)
ax.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# T_gibbs -- the non-circular half of the comparison.
#
# T_pca is by definition the fraction of an injected mock that survives a
# PCA clean. The equivalent for the sampler is the fraction that survives
# the Gibbs signal estimate. Comparing the two answers the actual claim --
# does marginalising over the foreground lose less signal than projecting
# it out -- and it does so without dividing any auto-power by T, so no
# residual or noise gets amplified.
#
# To use it, run the sampler TWICE with the same seeds and settings:
#   1. on data_cube            -> name_suffix = '_ref_data_'
#   2. on inj_ref (cell above) -> name_suffix = '_ref_inj_'
# mock_ref is saved to mock_ref_injection.npy so both arms and any later
# session use an identical injected signal.

def gibbs_signal_cube(suffix, n_samples, burn):
    """Posterior-mean signal cube from a run's x_sample traces."""
    acc = None
    for i in range(burn, n_samples):
        x = np.load(f'{out_folder}/samples/x_sample_{i}{suffix}.npy')
        sk = (x[0:rfft_len].reshape(rfft_shape)
              + 1j * x[rfft_len:2*rfft_len].reshape(rfft_shape))
        cube = Us(sk, False)
        acc = cube if acc is None else acc + cube
    return acc / (n_samples - burn)


try:
    s_data = gibbs_signal_cube('_ref_data_', N_total, cut)
    s_inj  = gibbs_signal_cube('_ref_inj_',  N_total, cut)
    mock_ref = np.load('mock_ref_injection.npy')
    T_gibbs  = pk_cross(s_inj - s_data, mock_ref) / pk_auto(mock_ref)

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    ax.plot(sig_k, T_m,      color='#eb6834', marker='s', lw=2, ms=5,
            label=f'$T_\\mathrm{{PCA}}$ ({nmodes_pca} modes)')
    ax.plot(sig_k, T_gibbs,  color='#2a78d6', marker='o', lw=2, ms=5,
            label='$T_\\mathrm{Gibbs}$')
    ax.axhline(1.0, color='0.4', lw=0.9, ls='--')
    ax.set_xscale('log'); ax.set_xlabel(r'k [Mpc$^{-1}$]')
    ax.set_ylabel('fraction of injected HI recovered')
    ax.set_title('Signal loss: marginalising vs projecting')
    ax.legend(frameon=False); ax.grid(alpha=0.15, lw=0.6)
    for side in ('top', 'right'): ax.spines[side].set_visible(False)
    plt.tight_layout(); plt.show()

    print(f'{"k":>10} {"T_PCA":>9} {"T_Gibbs":>9} {"ratio":>8}')
    for i in range(len(sig_k)):
        print(f'{sig_k[i]:10.4f} {T_m[i]:9.4f} {T_gibbs[i]:9.4f} '
              f'{T_gibbs[i]/T_m[i]:8.2f}')
except FileNotFoundError as e:
    print('injected/reference Gibbs runs not found yet --')
    print('  ', e)
    print('run the sampler on data_cube and on inj_ref with the')
    print('suffixes above, then re-run this cell.')


## P(k) inside the observed footprint

The `Pk_trace` files are the power of the **whole** reconstructed cube. 41% of those
voxels are inpainting — the sampler fills flagged regions from the prior, with no
data behind them — so `pk_mean` is a box average that mixes the observed sky with
prior draws. This recomputes $P(k)$ from the saved `x_sample_*` signal samples,
restricted to the footprint.

Restricting is multiplication in real space, i.e. convolution in $k$-space, so the
estimator needs the $\langle w^2\rangle$ normalisation *and* a check that it is
measuring the field rather than the mask. The check is built into the cell: mocks
with their fluctuations switched off outside the footprint are passed through
`pk_footprint`, which should return the input power. It does, to within ~10% in
every bin — so the difference below is a property of the reconstruction, not an
artefact of masking.

Two caveats. The lowest one or two $k$-bins are near the scale of the footprint
itself and carry large scatter, so treat them as indicative only. And the mask is
genuinely 3D, not a simple sky footprint: 1865 of 3150 pixels have some valid data
but only 923 are valid at every frequency, though those 1865 have a median of 499
of 500 good channels — so it is close to a per-pixel sky mask with a thin layer of
per-channel flagging on top.

In [ ]:
# Reuses mock(), pk_auto() and _unique_bins from the transfer-function cell above.

w2 = (flag**2).mean()          # = fill fraction, since flag is 0/1


def pk_footprint(a):
    """P(k) of a restricted to the observed footprint, normalised by <w^2>."""
    mu = a[flag == 1].mean()                       # mean over valid voxels only
    A = np.fft.fftn((a - mu) * flag, norm='ortho').flatten()
    p = (A * np.conj(A)).real
    return np.array([p[idxs == bb].mean() for bb in _unique_bins]) / w2


# ---- Is the estimator measuring the field or the mask? --------------------
# Take mocks and switch their fluctuations off outside the footprint, which is the
# regime the reconstruction is actually in (real structure inside, prior draws
# outside). pk_footprint should return the input power. If it does, any difference
# found below is a property of the reconstruction rather than mask leakage.
n_cal = 4
_cal = []
for _ in tqdm(range(n_cal), desc='mask calibration'):
    ms  = mock()
    _mu = ms.mean()
    _cal.append(pk_footprint(_mu + (ms - _mu) * flag) / pk_auto(ms))
mask_response = np.mean(_cal, axis=0)
print(f'mask response over {len(sig_k)} bins: '
      f'{mask_response.min():.2f} - {mask_response.max():.2f}  (want ~1)')

# ---- Recompute P(k) from the saved signal samples, both ways --------------
_pk_box, _pk_fp = [], []
for jj in tqdm(range(cut, N_total), desc='samples'):
    _x = np.load(out_folder + f'/samples/x_sample_{jj}' + name_suffix + '.npy')
    _s = Us(_x[:rfft_len].reshape(rfft_shape)
            + 1j * _x[rfft_len:2*rfft_len].reshape(rfft_shape), False).real
    _pk_box.append(pk_auto(_s))          # whole cube -- reproduces Pk_trace
    _pk_fp.append(pk_footprint(_s))      # observed footprint only

pk_box = np.mean(_pk_box, axis=0)
pk_fp  = np.mean(_pk_fp,  axis=0)

# Sanity check: the box-average recomputation should match the saved traces.
_agree = np.abs(pk_box / pk_mean - 1)[3:]     # skip the noisy lowest bins
print(f'recomputed box-average vs Pk_trace: max |ratio-1| = {_agree.max():.1e} '
      '(bins 3+)')

print(f'\n{"k [1/Mpc]":>10} {"box avg":>11} {"footprint":>11} {"ratio":>7} {"true HI":>11}')
for i in range(len(sig_k)):
    print(f'{sig_k[i]:10.4f} {pk_box[i]:11.3e} {pk_fp[i]:11.3e} '
          f'{pk_fp[i]/pk_box[i]:7.2f} {true_pk[i]:11.3e}')

In [ ]:
fig, (ax, axR) = plt.subplots(1, 2, figsize=(13, 5), dpi=200,
                              gridspec_kw={'width_ratios': [1.55, 1]})

# ---- left: the two ways of measuring the sampled signal ----
ax.plot(sig_k, pk_box, color='#2a78d6', ls='--', marker='o', lw=2, ms=5, zorder=4,
        label='Whole cube (what Pk_trace measures)')
ax.plot(sig_k, pk_fp, color='#eb6834', ls='-', marker='s', lw=2, ms=5, zorder=5,
        label='Observed footprint only')
ax.plot(sig_k, true_pk, '*-', color='#1baf7a', lw=2, ms=8, zorder=6,
        label=f'True HI (mean of {N_mocks} mocks)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'k [Mpc$^{-1}$]', fontsize=12)
ax.set_ylabel(r'P(k) [K$^2$]', fontsize=12)
ax.set_title('Sampled signal P(k): whole cube vs observed footprint', fontsize=12)
ax.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=8.5, loc='lower left')

# ---- right: the ratio, and the mask-response control ----
axR.plot(sig_k, pk_fp / pk_box, color='#eb6834', marker='s', lw=2, ms=5, zorder=3,
         label='Footprint / whole cube')
axR.plot(sig_k, mask_response, color='0.55', ls=':', marker='.', lw=1.5, ms=6,
         zorder=2, label='Mask-response control (want 1)')
axR.axhline(1.0, color='0.4', lw=0.9, ls='--', zorder=1)

axR.set_xscale('log')
axR.set_yscale('log')
axR.set_xlabel(r'k [Mpc$^{-1}$]', fontsize=12)
axR.set_ylabel('ratio', fontsize=12)
axR.set_title('Power inside the footprint, relative to the box', fontsize=12)
axR.grid(True, which='major', alpha=0.15, lw=0.6)
for side in ('top', 'right'):
    axR.spines[side].set_visible(False)
axR.legend(frameon=False, fontsize=8.5)

plt.tight_layout()
#plt.savefig('Pk_footprint_vs_box.png', bbox_inches='tight')
plt.show()